# SCRAPING


This notebook contains the code regarding the scrapping of the obituaries from https://www.currentobituary.com/ from 2002 to 2025. 

The scrapping part is huge and slow so, DON'T RUN THIS NOTEBOOK if you don't need to.

In [2]:
# Core imports
import pandas as pd
import time
import requests
from concurrent.futures import ThreadPoolExecutor

# For parsing HTML
from bs4 import BeautifulSoup

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

✅ All imports successful!


We run the following code 8 times, each one changing the batch of pages in order to get all the information needed.

In [ ]:
# We do it with different batches based on the pages of the website where the orbituaries that we want are
# 1 = [67,1050] 
# 2 = [1051, 2100] 
# 3 = [2101, 4100]
# 4 = [4101, 6000] 
# 5 = [6001, 8000]
# 6 = [8001, 10000]
# 7 = [10001, 11000]
# 8 = [11001, 13017]

PAGE_START = 4101 
PAGE_END = 6000    

def get_obituaries_from_page(page_num):
    base_url = "https://www.currentobituary.com"
    url = f"{base_url}/browse/p/{page_num}"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200: return []
        
        soup = BeautifulSoup(r.text, 'html.parser')
        items = []
        
        for el in soup.select(".username"):
            span = el.select_one("span")
            link = el.select_one("a")
            
            if span and link:
                date_txt = span.get_text().strip()
                href = link['href']
                
                full_url = href if href.startswith("http") else base_url + href
                
                items.append({
                    "url": full_url,
                    "date": date_txt
                })
        return items
    except:
        return []

def get_biography_fast(url):
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        container = soup.find(id="tab-obituary")
        if container:
            paragraphs = container.find_all("p")
            if paragraphs:
                txt = max([p.get_text().strip() for p in paragraphs], key=len)
                return txt if len(txt) > 50 else "Short text"
        return "Text not found"
    except:
        return "Error"   

def fetch_full_data(item):
    item['biography'] = get_biography_fast(item['url'])
    return item


# Execution

if __name__ == "__main__":
    all_links = []
        
    # Get URLs with requests 
    for p in range(PAGE_START, PAGE_END + 1):
        found = get_obituaries_from_page(p)
        if found:
            all_links.extend(found)
            print(f"Page {p}: Found {len(found)} cases. (Total: {len(all_links)})")
        
        if p % 10 == 0: time.sleep(1) 

    # Scrapping biographies
    if not all_links:
        print("Nothing was found.")
    else:
        print(f"\nStarting scraping of {len(all_links)} biographies.")
        final_results = []
        
        # Use ThreadPool to go faster
        with ThreadPoolExecutor(max_workers=10) as executor:
            final_results = list(executor.map(fetch_full_data, all_links))
        
        # Save the data
        df = pd.DataFrame(final_results)
        df.to_csv("obituaries_4_FINAL.csv", index=False, encoding='utf-8-sig')
        print("Done! File 'obituaries_4_FINAL.csv' generated.")

After running this we obtain 8 csv with all the data, so the next step is to create the final corpus of documents. To do so we create the final CSV by merging all the CSV files and applying the necessary filters (remove duplicates, remove years <2002 or >2025, remove entries with fewer than 100 characters, and remove the funeral section). 

In [ ]:
fitxers = [f'obituaries_{i}_FINAL.csv' for i in range(1, 9)]

cut_words = [
    "funeral service", "calling hours", "burial will be", 
    "interment", "cremation", "in lieu of flowers", 
    "memorial mass", "visitation will", "Directions to funeral"
]

def clean_cut_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Convert to lowercasing to make the search easier
    text_lower = text.lower()
    cut_posotion = len(text)
    
    # Find the first cut word
    for word in cut_words:
        index = text_lower.find(word.lower())
        if index != -1 and index < cut_posotion:
            cut_posotion = index
            
    # Cut the text
    return text[:cut_posotion].strip()

try:
    # Load and concatenate the files
    df = pd.concat([pd.read_csv(f) for f in fitxers], ignore_index=True)
    print(f"Initial number of documents {len(df)}")

    # Delete duplicates and dates out of range
    df = df.drop_duplicates(subset=['url'])
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # Apply cut text
    df['biography'] = df['biography'].apply(clean_cut_text)

    # Delete documents with lenght smaller than 100
    df = df[df['biography'].str.len() >= 100]

    # Delete documents with date out of range (2002-2025)
    df_final = df[
        (df['date'].dt.year >= 2002) & 
        (df['date'].dt.year <= 2025)
    ].copy()

    print(f"Number of documents after filtering: {len(df)}")

    # Save
    df_final.to_csv('obituaries.csv', index=False)

except Exception as e:
    print(f"Error: {e}")

We have finally obtained the corpus. We are saving it in the obituaries.csv file.